**Transformers From Scratch (Part 1): The Bridge Between Language and Maths**





The Story So Far...
We live in the era of Artificial Intelligence, dominated by massive models that can write code, compose poetry, and summarize textbooks in seconds. But beneath the polished interfaces of ChatGPT, Claude, and Llama lies a single, revolutionary architecture introduced by Google in 2017: The Transformer.

For many developers and data scientists, the Transformer remains a "black box." We know how to import it from Hugging Face in three lines of code, but what is actually happening under the hood? How does raw text turn into "understanding"?

**The Goal of This Series**

This series - Transformers From Scratch - is about breaking the glass. We are going to build the Transformer architecture completely from scratch using PyTorch. No black boxes, no hidden layers just pure code and intuition.

By the end of this journey, you will have a deep, mathematical, and programmatic understanding of how these engines work.

Our Roadmap will Start With:



*   Part 1: (We are Here) Tokenization & Input Embeddings
*   Part 2: Positional & Rotary Embeddings (RoPE)

*   Part 3: Self-Attention & Multi-Head Attention
*   Part 4: The Feed-Forward Network & Layer Normalization
*   Part 5: Assembling the Full Encoder Architecture

*   Part 6: Similar Journey for Decoder

Let's begin at step zero: getting our data ready.

-----------------------------------

**The Blank Canvas: Why Do We Need Tokenization?**

Imagine handing a dense novel to a calculator and asking it to find the main theme. The calculator would stare blankly at you because it only understands numbers, not the letters "A" through "Z."

Machine Learning models, at their core, are just highly complex calculators. They process matrices, perform dot products, and optimize gradients. They have zero concept of what a "word" is

*Tokenization is the bridge between human language and machine mathematics*.

-It is the process of breaking down raw, unstructured text into smaller, manageable chunks called "tokens," and then mapping each unique token to a specific integer ID. Before our model can learn relationships between words, it must first be able to read them.

*Text $\rightarrow$ Tokens $\rightarrow$ Integer IDs $\rightarrow$ Neural Network*

-------------------------

In [1]:
with open('alice_in_wonderland.txt','r',encoding = 'utf-8') as f:
  raw_text = f.read()

print("The Total Number of character is : ",len(raw_text))
print(raw_text[:210])

The Total Number of character is :  148208
TITLE: Alice's Adventures in Wonderland
AUTHOR: Lewis Carroll


= CHAPTER I = 
=( Down the Rabbit-Hole )=

  Alice was beginning to get very tired of sitting by her sister
on the bank, and of having nothing to 


Our goal is to tokenize this 148208 character short story into individual words and special
characters that we can then turn into embeddings for LLM training  

How can we split the text into tokens best with ? - regex library of Python

In [2]:
import re
text = "Hello World, I am Ayush and This is the Tokenization From Scratch. Is this a test -- ?"
result = re.split(r'(\s)',text)

print(result)

['Hello', ' ', 'World,', ' ', 'I', ' ', 'am', ' ', 'Ayush', ' ', 'and', ' ', 'This', ' ', 'is', ' ', 'the', ' ', 'Tokenization', ' ', 'From', ' ', 'Scratch.', ' ', 'Is', ' ', 'this', ' ', 'a', ' ', 'test', ' ', '--', ' ', '?']


The result is a list of individual words, whitespaces, and punctuation characters along with white spaces (It is optional to remove white spaces):

In [3]:
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', 'World', ',', 'I', 'am', 'Ayush', 'and', 'This', 'is', 'the', 'Tokenization', 'From', 'Scratch', '.', 'Is', 'this', 'a', 'test', '--', '?']


**The Whitespace Dilemma: To Strip or Not to Strip?**

When developing a simple tokenizer, the decision to include or remove whitespaces is a trade-off between structural precision and computational efficiency. Removing whitespaces is a common choice for introductory projects because it significantly reduces the sequence length and memory requirements, allowing us to focus purely on the relationship between semantic tokens (words and punctuation) without the added noise of "invisible" characters. However, while this approach ensures brevity and simplicity in our initial outputs, it does strip away the model's ability to learn specific formatting or indentation—a critical feature for tasks like code generation. For the purpose of this first stage, we will prioritize simplicity and remove whitespaces to keep our tokenized outputs clean, with the plan to transition to a more sophisticated, whitespace-aware scheme as our architecture grows in complexity.

Now Lets Apply this to Lewis Carroll's Alice in Wonderland book

In [4]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:50])

['TITLE', ':', 'Alice', "'", 's', 'Adventures', 'in', 'Wonderland', 'AUTHOR', ':', 'Lewis', 'Carroll', '=', 'CHAPTER', 'I', '=', '=', '(', 'Down', 'the', 'Rabbit-Hole', ')', '=', 'Alice', 'was', 'beginning', 'to', 'get', 'very', 'tired', 'of', 'sitting', 'by', 'her', 'sister', 'on', 'the', 'bank', ',', 'and', 'of', 'having', 'nothing', 'to', 'do', ':', 'once', 'or', 'twice', 'she']


In [5]:
print(len(preprocessed))

34158


**Creating a Token ID**

Once the text is preprocessed into individual tokens, our next task is to build a Vocabulary.

*A vocabulary is a unique set of every word and punctuation mark the model is "aware" of*. If a word isn't in this list, the model simply cannot "see" it.

We sort the tokens alphabetically for determinism. This ensures that every time we run our code, "Alice" always gets the same ID, and "Zebra" always gets the same ID. Without sorting, the internal mapping could change every time you restart your notebook, making your saved model weights useless.

In [6]:
# Using Set to remove duplicates automatically
unique_tokens = sorted(list(set(preprocessed)))

vocab_size = len(unique_tokens)

print(f"The total no. of Tokens in 'Alice in wonderland' : {len(preprocessed):,}")
print(f"Unique tokens (Vocabulary Size): {vocab_size:,}")

print(f"\nFirst 20 tokens in vocab: {unique_tokens[:20]}")

The total no. of Tokens in 'Alice in wonderland' : 34,158
Unique tokens (Vocabulary Size): 3,189

First 20 tokens in vocab: ['!', '"', "'", '(', ')', '*', ',', '--', '.', '0', '00', '10', '124', '1865-11-26', '2', '2021-03-08', '2021-08-03', '30', '5', '500']


The vocab_size is one of the most important hyperparameters in a Transformer. It defines the "width" of our final output layer. If our vocab size is 3,000, our model has to choose between 3,000 different possibilities every time it predicts the next word.

We need to create a "Translation Key" (Dictionaries) so we can quickly swap these strings for numbers and back again.

By using a simple Python dictionary, we create an $O(1)$ lookup table. Sorting our unique_tokens beforehand ensures that our mapping is deterministic—meaning every time you run this notebook, the word "Alice" will always be assigned the same ID.


In [7]:
stoi = {token: i for i, token in enumerate(unique_tokens)}
itos = {i: token for i, token in enumerate(unique_tokens)}

In [8]:
words = unique_tokens[:5]
print("Word -> ID Mapping:")
for word in words:
    print(f"  '{word}': {stoi[word]}")

Word -> ID Mapping:
  '!': 0
  '"': 1
  ''': 2
  '(': 3
  ')': 4


In [9]:
test_sentence = ["Alice", "was", "beginning", "to", "get", "very", "tired"]

encoded = [stoi.get(w) for w in test_sentence]
print(f"The Encoded sentence is : {encoded}")

decoded = [itos[i] for i in encoded]
print(f'The Decoded Sentence is : {decoded}')

The Encoded sentence is : [40, 3066, 913, 2904, 1589, 3039, 2901]
The Decoded Sentence is : ['Alice', 'was', 'beginning', 'to', 'get', 'very', 'tired']



Let's implement a complete tokenizer class in Python.
The class will have an encode method that splits
text into tokens and carries out the string-to-integer mapping to produce token IDs via the
vocabulary.
In addition, we implement a decode method that carries out the reverse
integer-to-string mapping to convert the token IDs back into text.


In [10]:
class SimpleTokenizerV1:
  def __init__(self,vocab):
    self.stoi = {token:i for i,token in enumerate(vocab)}
    self.itos = {i:token for i,token in enumerate(vocab)}

  def encode(self,text):

    preprocessed = re.split(r'([,.:;?_!"()'']|--|\s)',text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]

    ids = [self.stoi[s] for s in preprocessed]
    return ids

  def decode(self,ids):

    text = " ".join([self.itos[i] for i in ids])
    text = re.sub(r'\s+([,.?!"()''])', r'\1', text)
    return text


In [11]:
tokenizer = SimpleTokenizerV1(unique_tokens)

text =  "Hello,do you like the book?"
ids = tokenizer.encode(text)
print(f"Input Text: {text}")
print(f"Token IDs:  {ids}")

decoded = tokenizer.decode(ids)
print(f"Decoded:    {decoded}")

KeyError: 'Hello'

The Problem is the Word "Hello" was never used in our story. Hence it is not in our vocabulary. In the world of Large Language Models (LLMs), this is known as an Out-of-Vocabulary (OOV) error.

To build a model that doesn't crash when it sees a new word, we need to introduce Special Tokens. These are unique markers that carry specific structural meaning for the Transformer. We will now upgrade our vocabulary to include:

**ADDING SPECIAL CONTEXT TOKENS**

* <|unk|> (Unknown): This acts as a "catch-all" bucket. If the tokenizer encounters a word it hasn't seen before, it will map it to this ID instead of throwing an error.

* <|endoftext|> (End of Text): This serves as a boundary marker. When training on multiple unrelated documents, books, or articles, we insert this token to tell the model: "The previous story has ended; do not try to find a relationship between the last sentence and the next one."

Note: In GPT-like architectures, these tokens are crucial for maintaining context. Without <|endoftext|>, the model might think a cooking recipe and a news article are part of the same continuous thought!

In [24]:
unique_tokens = sorted(list(set(preprocessed)))

# Extend the vocabulary with our special tokens
# We add them to the list so they get their own unique IDs
unique_tokens.extend(["<|unk|>", "<|endoftext|>"])

vocab = {token:i for i,token in enumerate(unique_tokens)}

print(f"New Vocabulary Size: {len(vocab)}")
print(f"last 5 tokens: {unique_tokens[-5:]}")

New Vocabulary Size: 3191
last 5 tokens: ['yourself', 'youth', 'zigzag', '<|unk|>', '<|endoftext|>']


In [20]:
class SimpleTokenizerV2:

  def __init__(self, vocab):
      
    self.stoi = {token: i for i, token in enumerate(vocab)}
    self.itos = {i: token for i, token in enumerate(vocab)}

  def encode(self, text):
      
    preprocessed = re.split(r'([,.:;?_!"()'']|--|\s)',text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    preprocessed = [
        item if item in self.stoi
        else "<|unk|>" for item in preprocessed
    ]
      
    ids = [self.stoi[s] for s in preprocessed]
    return ids

  def decode(self, ids):
      
    text = " ".join([self.itos[i] for i in ids])
    text = re.sub(r'\s+([,.?!"()''])', r'\1', text)
    return text

In [23]:
tokenizer = SimpleTokenizerV2(vocab)

# Note: "Hello" and "Hatters" were NOT in our original tiny story sample
text_1 = "Hello, world!"
text_2 = "The Hatters had a tea party."
joined_text = f"<|endoftext|>".join([text_1, text_2]) # Inserting <|endoftext|>

print(f"Original Text: {joined_text}")

ids = tokenizer.encode(joined_text)
print(f"Token IDs:    {ids}")

decoded = tokenizer.decode(ids)
print(f"Decoded:      {decoded}")

Original Text: Hello, world!<|endoftext|>The Hatters had a tea party.
Token IDs:    [3189, 6, 3151, 0, 3189, 3189, 1649, 736, 2820, 2204, 8]
Decoded:      <|unk|>, world! <|unk|> <|unk|> had a tea party.


**Beyond Basic Tokens: Special Markers in LLMs**
While we have established that tokenization is the gateway for text entering an LLM, the process often involves more than just splitting words. Depending on the architecture, researchers use Special Tokens to provide the model with structural context.

Common Special Tokens
* [BOS] (Beginning of Sequence): Acts as a "start" signal, telling the model exactly where a new piece of content begins.

* [EOS] (End of Sequence): Essential for separating unrelated texts (like two different Wikipedia entries) within the same training block. It prevents the model from "hallucinating" a connection between them.

* [PAD] (Padding): Since GPUs love symmetry, we train models in batches. If one sentence has 10 tokens and another has 5, we use [PAD] to fill the gap so that every input in the batch has a uniform length.

**The GPT Exception: Simplified Efficiency**
Interestingly, the tokenizer used for GPT models (like GPT-2 or GPT-3) deviates from these standards for the sake of simplicity and robustness:

* The Universal Marker: Instead of separate BOS/EOS markers, GPT primarily uses the <|endoftext|> token to delimit sequences.

* No [UNK] (Unknown) Tokens: Most traditional tokenizers use an [UNK] tag for words they haven't seen before. GPT avoids this by using Byte Pair Encoding (BPE).

* Subword Logic: Because BPE breaks text down into subword units and eventually down to individual bytes if necessary, the model can represent any string of text. This effectively eliminates "Out-of-Vocabulary" (OOV) errors—the model will never encounter a word it literally cannot represent.